In [3]:
print(2+2)

4


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [18]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I would love to help you answer that, but I need a bit more context! **Which specific course are you referring to?**

Since "the course" could refer to anything from:
*   An online certification (e.g., Coursera, Udemy, LinkedIn Learning)
*   A university class
*   A private bootcamp or workshop
*   A local community class

Please share the **name of the course** or the **platform/school** where it’s being offered. Once you do, I can help you check:
1.  Enrollment deadlines
2.  Whether late registration is allowed
3.  Any prerequisites or costs to watch out for

I’m ready to help as soon as you give me the details! 😊


In [19]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [20]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [6]:
def llm(prompt):
    response = openai_client.responses.create(
        model="qwen/qwen3.8-27b",
        input=prompt
    )
    return response.output_text

In [21]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [8]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
if response.status_code == 200:
    courses_raw = response.json()
else:
    print(f"Failed to fetch courses. Status code: {response.status_code}")
    courses_raw = []

In [9]:
courses_raw

[{'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 470},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 65}]

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1427

In [11]:
documents[1000]

{'id': '5020067c2b',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 2. Machine Learning for Regression',
 'question': 'Standard Deviation Differences in Numpy and Pandas',
 'answer': 'Numpy and Pandas use different equations to compute the standard deviation. Numpy uses the population standard deviation by default, whereas Pandas uses the sample standard deviation.\n\n**Numpy**\n\n<{IMAGE:image_1}>\n\n**Pandas**\n\n<{IMAGE:image_2}>\n\nPandas computes the standard deviation using one degree of freedom by default. You can modify the degree of freedom in Numpy to achieve a similar result by using the `ddof` parameter:\n\n```python\nimport numpy as np\n\nnp.std(df.weight, ddof=1)\n```\n\nThe result will be similar if we set `ddof=1` in Numpy.'}

In [12]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [22]:
search_results = index.search(question,
            filter_dict={"course": "llm-zoomcamp"},
            num_results=5
            )

In [24]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {"course": course}

    return index.search(
            question,
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=5
            )

In [26]:
search_results = search(question)
#search_results

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [27]:
INSTRUCTIONS = ''''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
'''

In [28]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [29]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [33]:
def build_prompt(question, context):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
        )
    return prompt.strip()

In [34]:
prompt = build_prompt(question, search_results)

In [35]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w